# 04 — Linear regression from scratch

**Level:** Beginner · **Study time:** 60–75 minutes (including experiments)

**Prerequisites:** Notebooks 00–03.

## What you will be able to do
- Train your first predictive model.
- Separate training and test observations.
- Compare learned parameters with a known data-generating rule.

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 03](03_autograd.ipynb) · [Next: 05](05_neural_networks.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Generate noisy observations and split them

We simulate `y = 3x + 2 + noise`. Noise prevents perfect prediction. Split before training: the test set estimates performance on observations that did not influence learning. Here random splitting fits the independent toy observations.


In [ ]:
X = torch.linspace(-2, 2, 240).unsqueeze(1)
y = 3 * X + 2 + 0.25 * torch.randn_like(X)
order = torch.randperm(len(X))
train_idx, test_idx = order[:180], order[180:]
X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]
plt.scatter(X_train[:, 0], y_train[:, 0], s=12, label="train")
plt.scatter(X_test[:, 0], y_test[:, 0], s=12, label="test")
plt.legend(); plt.show()


## Step 3 — Write every part of the training loop

One iteration performs forward prediction, loss calculation, backward differentiation, and a parameter update. Loss is computed only on the training set. An **epoch** means one pass over all training examples; here every update uses the full training set.


In [ ]:
w = torch.randn(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
history = []
for epoch in range(180):
    prediction = X_train * w + b
    assert prediction.shape == y_train.shape
    loss = (prediction - y_train).square().mean()
    loss.backward()
    with torch.no_grad():
        w -= 0.08 * w.grad
        b -= 0.08 * b.grad
    w.grad = None
    b.grad = None
    history.append(loss.item())
print("Learned w, b:", w.item(), b.item())
assert history[-1] < history[0] * 0.1


## Step 4 — Evaluate against a baseline

A model should improve over a simple alternative. Our baseline always predicts the training target mean. Compute it from training data only, then compare both methods on the same test set.


In [ ]:
with torch.no_grad():
    test_pred = X_test * w + b
    test_mse = (test_pred - y_test).square().mean().item()
    baseline_mse = (y_train.mean() - y_test).square().mean().item()
print({"test_MSE": test_mse, "mean_baseline_MSE": baseline_mse})
assert test_mse < baseline_mse
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(history); ax[0].set(title="Learning curve", xlabel="Epoch", ylabel="Train MSE")
ax[1].scatter(X_test[:, 0], y_test[:, 0], s=12)
ax[1].plot(X[:, 0], (X * w + b).detach()[:, 0], color="orange")
ax[1].set(title="Held-out predictions", xlabel="x", ylabel="y")
plt.tight_layout(); plt.show()


## Your turn — practice before opening the solutions

1. Compute test mean absolute error (MAE).
2. Fit a second model using `torch.optim.SGD` instead of manual updates.
3. Compare both models' parameters within a small tolerance. Hint: initialize the second model at zero and train for 200 steps.

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
with torch.no_grad():
    print("Test MAE:", (test_pred - y_test).abs().mean().item())
w2 = torch.zeros(1, requires_grad=True)
b2 = torch.zeros(1, requires_grad=True)
optimizer = torch.optim.SGD([w2, b2], lr=0.08)
for _ in range(200):
    optimizer.zero_grad(set_to_none=True)
    ((X_train * w2 + b2 - y_train) ** 2).mean().backward()
    optimizer.step()
assert torch.allclose(w2, w, atol=1e-3)
assert torch.allclose(b2, b, atol=1e-3)


## Before moving on

Training error measures fit to seen examples; test error estimates generalization. If you repeatedly choose settings using test error, that set has effectively become a validation set.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [Learning PyTorch with examples](https://docs.pytorch.org/tutorials/beginner/pytorch_with_examples.html)


[Course guide](../README.md) · [Previous: 03](03_autograd.ipynb) · [Next: 05](05_neural_networks.ipynb)
